In [0]:
%run ./00_config_schemas

In [0]:
import uuid, json
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def create_tables():
    for name, sch in [(T_BRONZE, BRONZE_SCHEMA_DEF), (T_SILVER, SILVER_SCHEMA_DEF),
                      (T_QUAR, QUAR_SCHEMA_DEF), (T_LOGS, LOG_SCHEMA_DEF)]:
        (spark.createDataFrame([], sch).write.format("delta")
              .mode("ignore").saveAsTable(name))

def new_batch_id(prefix="batch"):
    return f"{prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"

def merge_metrics(table):
    h = (DeltaTable.forName(spark, table).history(1)
         .select("operationMetrics").collect()[0][0])
    return int(h.get("numTargetRowsInserted", 0)), int(h.get("numTargetRowsUpdated", 0))

def log_run(run_id, layer, params, start, end, status, ins=0, upd=0, err=None):
    row = [(run_id, layer, str(params), start, end, status, int(ins), int(upd), err)]
    (spark.createDataFrame(row, LOG_SCHEMA_DEF)
          .write.format("delta").mode("append").saveAsTable(T_LOGS))

In [0]:
def _f(x):
    return None if x is None else float(x)

def bronze_load(path, ltype, bid):
    rid, t0 = new_batch_id("run"), datetime.now()
    try:
        d = json.load(open(path))
        ok, bad = [], []
        for r in d["data"]:
            keys_ok = set(r) == EXP_KEYS
            null_ok = (r.get("coin_id") is not None and
                       r.get("timestamp") is not None and
                       r.get("price") is not None)
            if keys_ok and null_ok:
                ok.append((r["coin_id"], int(r["timestamp"]), _f(r["price"]),
                           _f(r["market_cap"]), _f(r["total_volume"])))
            else:
                why = ("schema drift: extra=%s missing=%s"
                       % (sorted(set(r) - EXP_KEYS), sorted(EXP_KEYS - set(r)))
                       if not keys_ok else "null key/price")
                bad.append((json.dumps(r), why))

        now = F.current_timestamp()
        src = (spark.createDataFrame(ok, RAW_REC_SCHEMA)
               .dropDuplicates(["coin_id", "ts_ms"])
               .withColumn("source_file", F.lit(path))
               .withColumn("load_type", F.lit(ltype))
               .withColumn("batch_id", F.lit(bid))
               .withColumn("load_timestamp", now))
        src.createOrReplaceTempView("src_b")

        spark.sql(f"""
          MERGE INTO {T_BRONZE} t USING src_b s
          ON t.coin_id = s.coin_id AND t.ts_ms = s.ts_ms
          WHEN MATCHED AND NOT (t.price <=> s.price
                                AND t.market_cap <=> s.market_cap
                                AND t.total_volume <=> s.total_volume)
            THEN UPDATE SET *
          WHEN NOT MATCHED THEN INSERT *""")
        ins, upd = merge_metrics(T_BRONZE)

        if bad:
            (spark.createDataFrame(bad, "raw_record string, reason string")
             .withColumn("source_file", F.lit(path))
             .withColumn("batch_id", F.lit(bid))
             .withColumn("load_timestamp", now)
             .createOrReplaceTempView("q_src"))
            spark.sql(f"""
              MERGE INTO {T_QUAR} t USING q_src s
              ON t.raw_record = s.raw_record AND t.source_file = s.source_file
              WHEN NOT MATCHED THEN INSERT *""")

        log_run(rid, "bronze", f"{path} | batch={bid} | bad={len(bad)}",
                t0, datetime.now(), "SUCCESS", ins, upd)
        print(f"read={len(d['data'])} good={len(ok)} quarantined={len(bad)} "
              f"inserted={ins} updated={upd}")
    except Exception as e:
        log_run(rid, "bronze", f"{path} | batch={bid}",
                t0, datetime.now(), "FAILED", err=str(e)[:500])
        raise

In [0]:
create_tables()
display(spark.sql("SHOW TABLES IN crypto.bronze"))
display(spark.sql("SHOW TABLES IN crypto.silver"))
display(spark.sql("SHOW TABLES IN crypto.ops"))